In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from scipy.stats import mode
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Data Loading

In [40]:
df = pd.read_csv("student_performance_rf.csv")
df.head()

,Age,Gender,StudyHours,Attendance,Assignments,Midterm,Final,Participation,Internet,Result
0,18,M,2,75,60,55,58,50,Yes,Fail
1,19,F,5,90,85,78,82,88,Yes,Pass
2,20,M,1,60,40,35,30,45,No,Fail
3,21,F,6,95,90,88,91,92,Yes,Pass
4,22,M,3,80,70,65,68,72,Yes,Pass


In [41]:
print("Shape:", df.shape)

print("\nMissing Values:\n", df.isnull().sum())

print("\nTarget Distribution:\n", df['Result'].value_counts())

Shape: (30, 10)

Missing Values:
 Age              0
Gender           0
StudyHours       0
Attendance       0
Assignments      0
Midterm          0
Final            0
Participation    0
Internet         0
Result           0
dtype: int64

Target Distribution:
 Result
Pass    19
Fail    11
Name: count, dtype: int64


## Data Preprocessing
- Encode categorical variables
- Split dataset

In [42]:
le_gender = LabelEncoder()
le_internet = LabelEncoder()
le_result = LabelEncoder()

df['Gender'] = le_gender.fit_transform(df['Gender'])
df['Internet'] = le_internet.fit_transform(df['Internet'])
df['Result'] = le_result.fit_transform(df['Result'])

In [43]:
X = df.drop('Result', axis=1)
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Random Forest Model

In [44]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [45]:
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 1.0

Confusion Matrix:
 [[3 0]
 [0 3]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6



## Custom Bagging Model
Using Decision Trees as base learners

In [46]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [47]:
def bagger_fit_predict(model_fns, X_train, y_train, X_test, n_estimators=15):
    all_preds = []

    for fn in model_fns:
        for _ in range(n_estimators):
            # Bootstrap sampling
            X_bs, y_bs = resample(X_train, y_train, replace=True)

            # Create model
            model = fn()

            # Train
            model.fit(X_bs, y_bs)

            # Predict
            preds = model.predict(X_test)
            all_preds.append(preds)

    all_preds = np.array(all_preds)

    # Majority voting
    final_preds = mode(all_preds, axis=0).mode

    return final_preds.ravel()

In [48]:
model_list = [
    lambda: DecisionTreeClassifier(max_depth=5),
    lambda: KNeighborsClassifier(n_neighbors=5),
    lambda: LogisticRegression(max_iter=7000)
]

In [49]:
bagger_preds = bagger_fit_predict(
    model_fns=model_list,
    X_train=X_train_scaled,   # IMPORTANT: use scaled
    y_train=y_train,
    X_test=X_test_scaled,
    n_estimators=15
)

bagger_acc = accuracy_score(y_test, bagger_preds)

print("Custom Bagger Accuracy:", bagger_acc)

Custom Bagger Accuracy: 1.0


## Comparison

- Random Forest uses multiple decision trees with feature randomness.
- Bagging uses multiple models trained on bootstrapped samples.

### Observations:
- Compare accuracy
- Compare precision, recall, f1-score
- Check confusion matrices